<a href="https://colab.research.google.com/github/DanilaKrug/ai_miit/blob/practice-2/notebooks/02_tool_calling_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практика 2. Базовый tool-calling агент

**Дисциплина:** Генеративный ИИ и ИИ-агенты для транспортной логистики (РУТ (МИИТ), магистратура)
**Время:** 90 минут. **Среда:** Google Colab.

## Что мы сегодня делаем

На прошлом занятии вы научили модель превращать заявку в структуру. Сегодня даём ей
**инструменты** — и пишем цикл, который эти инструменты вызывает.

К концу занятия у вас будет `run_agent()`: задаёте вопрос по данным вагонных перевозок
словами — агент сам решает, какой инструмент позвать, с какими аргументами, получает
число и отвечает человеку.

> «Сколько вагонов выгрузили на дороге ЛСН в марте 2021?»
> → агент зовёт `query_unload(date_from="2021-03-01", date_to="2021-03-31", road="ЛСН")`
> → получает 9 761 → отвечает с указанием периода и фильтров.

**Главное, что надо унести с занятия:** модель не вызывает инструменты. Она возвращает
**запрос на вызов** — имя и аргументы. Вызываете вы, вот этим кодом.

**Критерий сдачи:** зелёный `check()` в конце — 6 проверок из 6.


---
## 0. Установка зависимостей

В чистом Colab достаточно одной строки ниже. Локально — то же самое в своём venv.


In [30]:
!pip install -q "openai>=1.40,<2" "pandas>=2.0"


---
## 1. Учебные данные

Три витрины из обезличенного набора курса — те же строки, что в `data/dataset_student.zip`,
только без колонок, которые сегодня не нужны, и без времени в отчётной дате:

| Файл | Что | Строк |
|---|---|---|
| `unload_slice.csv.gz` | выгрузка: дата, станция, дорога, род вагона, клиент, груз | 170 998 |
| `load_slice.csv.gz` | погрузка, те же измерения | 398 130 |
| `disloc_slice.csv.gz` | дислокация: где какой вагон стоит **сейчас** | 4 553 |

Период данных — **2020-11-01 … 2021-04-30**. «Сегодня» курса — **2021-04-30**.

Дислокация — это **текущий срез**, а не история: вопрос «где вагон стоял в феврале»
корректного ответа по ней не имеет. Это пригодится дальше.


In [31]:
import json
import os
from pathlib import Path

import pandas as pd

# Каталог notebooks/data в публичном репозитории курса.
COURSE_DATA_URL = "https://raw.githubusercontent.com/MaximSantalov/ai_miit/main/notebooks/data"
DATA_FILES = ("unload_slice.csv.gz", "load_slice.csv.gz", "disloc_slice.csv.gz",
              "agent_facts.json")


def _find_local_data_dir():
    candidates = [os.environ.get("COURSE_DATA_DIR"), "notebooks/data", "data", "../data", "."]
    for c in candidates:
        if not c:
            continue
        p = Path(c)
        if all((p / f).exists() for f in DATA_FILES):
            return p
    return None


def ensure_data() -> Path:
    """Возвращает каталог с учебными данными, при необходимости скачивая их."""
    local = _find_local_data_dir()
    if local is not None:
        return local

    import urllib.request

    target = Path("data")
    target.mkdir(exist_ok=True)
    for f in DATA_FILES:
        try:
            urllib.request.urlretrieve(f"{COURSE_DATA_URL}/{f}", target / f)
        except Exception as e:  # noqa: BLE001
            raise RuntimeError(
                f"Не удалось скачать {f} из репозитория курса ({type(e).__name__}: {e}).\n"
                f"Что сделать: проверьте COURSE_DATA_URL в этой ячейке или положите файлы "
                f"{list(DATA_FILES)} рядом с ноутбуком в каталог data/ вручную."
            ) from e
    return target


# Измерения читаем категориями: так три витрины занимают десятки мегабайт, а не сотни.
_CAT = "category"
DATA_DIR = ensure_data()

UNLOAD = pd.read_csv(
    DATA_DIR / "unload_slice.csv.gz",
    dtype={"REPDATE": str, "ST_NAME": _CAT, "RW_NAME": _CAT, "ROD_NAME": _CAT,
           "CLIENT_NAME": _CAT, "GG_NAME": _CAT, "UNLOADED_WAGONS": int},
)
LOAD = pd.read_csv(
    DATA_DIR / "load_slice.csv.gz",
    dtype={"REPDATE": str, "ST_NAME": _CAT, "RW_NAME": _CAT, "ROD_NAME": _CAT,
           "CLIENT_NAME": _CAT, "GG_NAME": _CAT, "LOADED_WAGONS": int},
)
DISLOC = pd.read_csv(DATA_DIR / "disloc_slice.csv.gz", dtype=str).fillna("")
FACTS = json.loads((DATA_DIR / "agent_facts.json").read_text(encoding="utf-8"))

TODAY = "2021-04-30"                     # «сегодня» курса — конец учебного периода
ROADS = sorted(FACTS["roads"])           # 10 дорог набора
RODS = sorted(UNLOAD["ROD_NAME"].dropna().unique())

print(f"выгрузка:   {len(UNLOAD):>7} строк, {UNLOAD.REPDATE.min()} … {UNLOAD.REPDATE.max()}")
print(f"погрузка:   {len(LOAD):>7} строк")
print(f"дислокация: {len(DISLOC):>7} строк — текущий срез на {TODAY}")
print(f"дороги: {', '.join(ROADS)}")
print(f"рода вагонов: {', '.join(RODS)}")


выгрузка:    170998 строк, 2020-11-01 … 2021-04-30
погрузка:    398130 строк
дислокация:    4553 строк — текущий срез на 2021-04-30
дороги: ГРН, ДОЛ, ЗАЛ, КРЖ, ЛСН, ОЗР, ПРБ, ПУЩ, СТП, ТУН
рода вагонов: КР, ПВ, ПЛ, ПЛФИТ, ЦМВ, ЦС


In [32]:
import zipfile
import urllib.request

GU12_ZIP_URL = "https://raw.githubusercontent.com/MaximSantalov/ai_miit/main/data/dataset_student.zip"
GU12_LOCAL_ZIP = Path("data/dataset_student.zip")
GU12_CSV_NAME = "dataset_student/V_DM_GU12.csv"

if not GU12_LOCAL_ZIP.exists():
    GU12_LOCAL_ZIP.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(GU12_ZIP_URL, GU12_LOCAL_ZIP)

with zipfile.ZipFile(GU12_LOCAL_ZIP) as z:
    with z.open(GU12_CSV_NAME) as f:
        GU12 = pd.read_csv(
            f,
            dtype={"ST_NAME": _CAT, "RW_NAME": _CAT, "ROD_NAME": _CAT, "GG_NAME": _CAT,
                   "GU12_WAG_CNT": int},
        )

# REPDATE тут с временем ("2021-03-25 12:17:19") — обрезаем до даты для группировки по дням
GU12["REPDATE"] = GU12["REPDATE"].str.slice(0, 10)

print(f"ГУ-12: {len(GU12):>7} строк, {GU12.REPDATE.min()} … {GU12.REPDATE.max()}")

ГУ-12:  112888 строк, 2020-11-01 … 2021-04-30


In [33]:
def tool_query_gu12(date_from=None, date_to=None, road=None, station=None, rod=None) -> dict:
    date_from, date_to = _period(date_from, date_to)
    road = _one_of(road, set(ROADS), "Дорога")
    rod = _one_of(rod, set(RODS), "Род подвижного состава")
    station = _one_of(station, set(GU12["ST_NAME"].cat.categories), "Станция", show=False)

    df = GU12
    used = {}
    if road:
        df, used["road"] = df[df["RW_NAME"] == road], road
    if station:
        df, used["station"] = df[df["ST_NAME"] == station], station
    if rod:
        df, used["rod"] = df[df["ROD_NAME"] == rod], rod

    mask = (df["REPDATE"] >= date_from) & (df["REPDATE"] <= date_to)
    sub = df.loc[mask]

    daily = sub.groupby("REPDATE", observed=True)["GU12_WAG_CNT"].sum()
    days_covered = len(daily)
    avg = round(float(daily.mean()), 1) if days_covered else 0.0

    return {
        "metric": "среднесуточное количество вагонов в заявках ГУ-12",
        "period": [date_from, date_to],
        "days_covered": days_covered,
        "wagons_avg_per_day": avg,
        "filters": used,
        "note": ("Это среднее по дням, а не сумма: заявки ГУ-12 — снимок остатка, "
                 "а не поток вагонов, складывать дни некорректно."),
    }

---
## 2. Подключение к курсовому прокси

Всё как на практике 1: ходим в **курсовой LiteLLM-прокси**, он совместим с OpenAI API,
поэтому пользуемся библиотекой `openai` с другим `base_url`.

На прокси два имени моделей: `course-fast` и `course-smart`. Начинаем с `course-fast`;
если она не умеет вызывать инструменты — следующая ячейка это скажет прямым текстом,
и вы поменяете одну строку здесь.


In [34]:
import getpass

from openai import OpenAI

# ── Куда ходим и какой моделью ──────────────────────────────────────────
# Вариант 1 (по умолчанию): курсовой ключ и курсовой прокси.
BASE_URL = "https://llm.kamani.tech/v1"
MODEL = "course-fast"

# Вариант 2: сюда переключаемся, если course-fast не умеет инструменты.
# MODEL = "course-smart"
#
# Вариант 3: у вас свой ключ DeepSeek или OpenAI.
# BASE_URL = "https://api.deepseek.com/v1"
# MODEL = "deepseek-chat"
# BASE_URL = "https://api.openai.com/v1"
# MODEL = "gpt-4o-mini"
#
# Больше в ноутбуке имя модели нигде не зашито: везде используется MODEL.


def connect(reset: bool = False) -> OpenAI:
    """Создаёт клиента. connect(reset=True) — ввести ключ заново."""
    global client
    if reset:
        os.environ.pop("COURSE_LLM_API_KEY", None)
    if not os.environ.get("COURSE_LLM_BASE_URL"):
        os.environ["COURSE_LLM_BASE_URL"] = (
            BASE_URL or input("base_url (например https://.../v1): ").strip()
        )
    if not os.environ.get("COURSE_LLM_API_KEY"):
        os.environ["COURSE_LLM_API_KEY"] = getpass.getpass("Ваш ключ (ввод не виден): ").strip()
    client = OpenAI(
        base_url=os.environ["COURSE_LLM_BASE_URL"],
        api_key=os.environ["COURSE_LLM_API_KEY"],
    )
    print("Клиент создан. base_url =", os.environ["COURSE_LLM_BASE_URL"], "| модель:", MODEL)
    return client


client = connect()


Клиент создан. base_url = https://llm.kamani.tech/v1 | модель: course-fast


---
## 3. Преполётная проверка: умеет ли модель вызывать инструменты

Вызов инструментов — это отдельная способность модели, а не свойство API. Модель может
прекрасно писать текст и при этом игнорировать переданные ей инструменты.

Ячейка ниже делает один пробный запрос с игрушечным инструментом и говорит прямо:
поддерживается или нет. **Не пропускайте её** — без этого весь остальной ноутбук
будет молча возвращать текст вместо вызовов.


In [35]:
PROBE_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_wagons_on_station",
        "description": "Возвращает количество вагонов на станции прямо сейчас.",
        "parameters": {
            "type": "object",
            "properties": {"station": {"type": "string", "description": "Название станции"}},
            "required": ["station"],
        },
    },
}]


def check_tools(model: str = MODEL) -> bool:
    """Проверяет, умеет ли модель возвращать запрос на вызов инструмента."""
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user",
                       "content": "Сколько вагонов сейчас на станции ДОЛИНО? "
                                  "Ответь, воспользовавшись инструментом."}],
            tools=PROBE_TOOL,
            tool_choice="auto",
            max_tokens=200,
        )
    except Exception as e:  # noqa: BLE001
        text = f"{type(e).__name__}: {e}"
        print(f"❌ Пробный запрос с инструментом не прошёл: {text}")
        print("   Если в тексте выше есть 'tools' или 'function' — модель их не принимает.")
        print("   Что делать: в ячейке выше поменяйте MODEL на 'course-smart', "
              "выполните её заново и повторите эту проверку.")
        return False

    calls = r.choices[0].message.tool_calls
    if not calls:
        print(f"❌ Модель '{model}' не вернула запрос на вызов — она ответила текстом:")
        print("  ", (r.choices[0].message.content or "").strip()[:300])
        print("   Что делать: в ячейке выше поменяйте MODEL на 'course-smart', "
              "выполните её заново и повторите эту проверку. "
              "Остальной ноутбук при этом не меняется.")
        return False

    call = calls[0]
    print(f"✅ Модель '{model}' умеет вызывать инструменты.")
    print(f"   Она попросила вызвать: {call.function.name}({call.function.arguments})")
    print("   Обратите внимание: данных в ответе нет. Есть только просьба.")
    return True


check_tools()


✅ Модель 'course-fast' умеет вызывать инструменты.
   Она попросила вызвать: get_wagons_on_station({"station": "ДОЛИНО"})
   Обратите внимание: данных в ответе нет. Есть только просьба.


True

---
## 4. Как этот ноутбук сообщает о невыполненных заданиях

Как и на практике 1: в ячейках с заданием стоит пустое место с подписью «здесь ваш ответ».
Пока оно пустое, готовый код это замечает сам и печатает строку о том, какое задание
ещё не сделано, — вместо трассировки на пол-экрана.


In [36]:
class TodoNotDone(NotImplementedError):
    """Помеченная ячейка # TODO ещё не дописана: функция ничего не вернула."""


def need(value, number: int, what: str):
    """Проверяет результат вашей функции. Готовый код, менять не нужно."""
    if value is None or (isinstance(value, (str, list, dict)) and len(value) == 0):
        raise TodoNotDone(f"TODO {number} не выполнен: {what}")
    return value


def explain_todo(e: TodoNotDone) -> None:
    """Печатает понятное сообщение вместо трассировки."""
    print("⛔", e)
    print("   Что делать: допишите ячейку с заданием выше, выполните её (Shift+Enter)")
    print("   и запустите эту ячейку заново.")


---
## 5. Три инструмента: функции уже написаны

Инструмент агента — это обычная функция. Никакой магии в ней нет: аргументы, фильтрация,
число на выходе.

Эти три написаны за вас — сегодня вы пишете не pandas, а **шов между моделью и кодом**.
Прочитайте их сигнатуры: именно их вы будете описывать в TODO 1, и модель увидит ровно
то, что вы напишете.

Обратите внимание на две вещи, они понадобятся дальше:

* **инструмент проверяет аргументы и падает с понятным текстом** — «дороги ВСТ в наборе
  нет, есть такие-то». Это не сбой, это сообщение, которое агент сможет прочитать;
* **`find_wagons` работает только с текущим срезом** и говорит об этом в ответе.


In [37]:
MAX_ROWS = 40          # больше строк в ответ инструмента не кладём: это контекст и деньги
MAX_WAGONS_LOOKUP = 20  # столько номеров вагонов можно спросить за раз


def _period(date_from, date_to):
    """Проверяет период и возвращает пару дат. Без периода считать нельзя."""
    if not date_from or not date_to:
        raise ValueError(
            "Не указан период. Нужны обе даты: date_from и date_to в формате ГГГГ-ММ-ДД. "
            f"Данные есть за 2020-11-01 … {TODAY}."
        )
    if str(date_from) > str(date_to):
        raise ValueError(f"Период задом наперёд: date_from={date_from} позже date_to={date_to}.")
    return str(date_from)[:10], str(date_to)[:10]


def _one_of(value, allowed, what: str, show: bool = True):
    """Сверяет значение со справочником. None и пустая строка — значит фильтра нет."""
    if value is None or str(value).strip() == "":
        return None
    v = str(value).strip().upper()
    if v not in allowed:
        hint = f" Допустимые значения: {', '.join(sorted(allowed))}." if show else ""
        raise ValueError(f"{what}: значения '{value}' в наборе нет.{hint}")
    return v


def _aggregate(df, date_col, value_col, date_from, date_to, group_by, label):
    mask = (df[date_col] >= date_from) & (df[date_col] <= date_to)
    sub = df.loc[mask]
    answer = {"metric": label, "period": [date_from, date_to], "wagons": int(sub[value_col].sum())}
    if group_by in ("day", "month"):
        key = sub[date_col] if group_by == "day" else sub[date_col].str.slice(0, 7)
        grouped = sub.groupby(key, observed=True)[value_col].sum().sort_index()
        rows = [{"period": str(k), "wagons": int(v)} for k, v in grouped.items()]
        answer["by_" + group_by] = rows[:MAX_ROWS]
        if len(rows) > MAX_ROWS:
            answer["note"] = f"показаны первые {MAX_ROWS} из {len(rows)} строк"
    return answer


def tool_query_unload(date_from=None, date_to=None, road=None, station=None,
                      rod=None, group_by=None) -> dict:
    """Сколько вагонов выгружено за период. Только чтение."""
    date_from, date_to = _period(date_from, date_to)
    road = _one_of(road, set(ROADS), "Дорога")
    rod = _one_of(rod, set(RODS), "Род подвижного состава")
    station = _one_of(station, set(UNLOAD["ST_NAME"].cat.categories), "Станция", show=False)

    df = UNLOAD
    used = {}
    if road:
        df, used["road"] = df[df["RW_NAME"] == road], road
    if station:
        df, used["station"] = df[df["ST_NAME"] == station], station
    if rod:
        df, used["rod"] = df[df["ROD_NAME"] == rod], rod

    answer = _aggregate(df, "REPDATE", "UNLOADED_WAGONS", date_from, date_to,
                        group_by, "выгружено вагонов")
    answer["filters"] = used
    return answer


def tool_query_load(date_from=None, date_to=None, road=None, station=None,
                    rod=None, group_by=None) -> dict:
    """Сколько вагонов погружено за период. Только чтение."""
    date_from, date_to = _period(date_from, date_to)
    road = _one_of(road, set(ROADS), "Дорога")
    rod = _one_of(rod, set(RODS), "Род подвижного состава")
    station = _one_of(station, set(LOAD["ST_NAME"].cat.categories), "Станция", show=False)

    df = LOAD
    used = {}
    if road:
        df, used["road"] = df[df["RW_NAME"] == road], road
    if station:
        df, used["station"] = df[df["ST_NAME"] == station], station
    if rod:
        df, used["rod"] = df[df["ROD_NAME"] == rod], rod

    answer = _aggregate(df, "REPDATE", "LOADED_WAGONS", date_from, date_to,
                        group_by, "погружено вагонов")
    answer["filters"] = used
    return answer


def tool_find_wagons(wagon_numbers=None, station=None, road=None) -> dict:
    """Где вагоны находятся СЕЙЧАС. Истории перемещений в этом источнике нет."""
    note = (f"текущий срез дислокации на {TODAY}; "
            f"историю перемещений по нему получить нельзя")

    if wagon_numbers:
        if isinstance(wagon_numbers, (str, int)):
            wagon_numbers = [wagon_numbers]
        wagon_numbers = [str(w).strip() for w in wagon_numbers]
        if len(wagon_numbers) > MAX_WAGONS_LOOKUP:
            raise ValueError(
                f"За один раз можно спросить не больше {MAX_WAGONS_LOOKUP} номеров, "
                f"передано {len(wagon_numbers)}."
            )
        found, missing = [], []
        for num in wagon_numbers:
            rows = DISLOC[DISLOC["CARNUM"] == num]
            if rows.empty:
                missing.append(num)
                continue
            r = rows.iloc[0]
            found.append({"wagon": num, "station": r["ST_DISL_NAME"], "road": r["ROAD_DISL"],
                          "state": r["LOADED"], "operation": r["OPER_NAME"],
                          "park": r["PARK_SIGN"]})
        return {"as_of": TODAY, "found": found, "not_found": missing, "note": note}

    road = _one_of(road, set(DISLOC["ROAD_DISL"].unique()), "Дорога")
    station = _one_of(station, set(DISLOC["ST_DISL_NAME"].unique()), "Станция", show=False)
    if not road and not station:
        raise ValueError("Нужен либо список номеров вагонов, либо станция, либо дорога.")

    df = DISLOC
    used = {}
    if road:
        df, used["road"] = df[df["ROAD_DISL"] == road], road
    if station:
        df, used["station"] = df[df["ST_DISL_NAME"] == station], station
    return {"as_of": TODAY, "filters": used, "wagons": int(len(df)), "note": note}


### Проверьте, что функции работают

Ячейка готова, ничего дописывать не нужно. Она вызывает инструменты **напрямую из Python** —
никакой модели здесь ещё нет. Это важный момент: инструменты живут отдельно и работают сами
по себе, модель им не нужна.


In [38]:
print("выгрузка ЛСН за март 2021:")
print("  ", tool_query_unload(date_from="2021-03-01", date_to="2021-03-31", road="ЛСН"))

print("\nвагоны на дороге ЗАЛ прямо сейчас:")
print("  ", tool_find_wagons(road="ЗАЛ"))

print("\nа так инструмент отказывается работать (дороги ВСТ в наборе нет):")
try:
    tool_query_unload(date_from="2021-03-01", date_to="2021-03-31", road="ВСТ")
except ValueError as e:
    print("   ValueError:", e)


выгрузка ЛСН за март 2021:
   {'metric': 'выгружено вагонов', 'period': ['2021-03-01', '2021-03-31'], 'wagons': 9761, 'filters': {'road': 'ЛСН'}}

вагоны на дороге ЗАЛ прямо сейчас:
   {'as_of': '2021-04-30', 'filters': {'road': 'ЗАЛ'}, 'wagons': 509, 'note': 'текущий срез дислокации на 2021-04-30; историю перемещений по нему получить нельзя'}

а так инструмент отказывается работать (дороги ВСТ в наборе нет):
   ValueError: Дорога: значения 'ВСТ' в наборе нет. Допустимые значения: ГРН, ДОЛ, ЗАЛ, КРЖ, ЛСН, ОЗР, ПРБ, ПУЩ, СТП, ТУН.


---
## 6. TODO 1 — описать инструменты для модели

Модель не видит ваш Python. Она видит **только** то, что вы про инструмент написали:
имя, описание и схему аргументов. Описание инструмента — это промпт, и работает он
ровно так же: чем точнее, тем меньше глупостей.

Формат — тот, что понимает OpenAI-совместимый API (наш прокси в том числе):

```python
{
  "type": "function",
  "function": {
      "name": "имя_как_в_питоне",
      "description": "что делает и когда звать",
      "parameters": {  # это обычная JSON-схема
          "type": "object",
          "properties": {"аргумент": {"type": "string", "description": "..."}},
          "required": ["обязательные", "аргументы"],
      },
  },
}
```

Первый инструмент описан за вас целиком — разберите его построчно, дальше по образцу.


In [39]:
# ── Готовый образец: описание первого инструмента ──────────────────────
TOOL_QUERY_UNLOAD = {
    "type": "function",
    "function": {
        "name": "query_unload",
        "description": (
            "Сколько вагонов ВЫГРУЖЕНО за период. Обязательно нужен период — две даты. "
            "Можно сузить по дороге, станции выгрузки и роду подвижного состава. "
            "Для погрузки есть отдельный инструмент query_load — этот про выгрузку."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "date_from": {"type": "string",
                              "description": "начало периода, ГГГГ-ММ-ДД"},
                "date_to": {"type": "string",
                            "description": "конец периода включительно, ГГГГ-ММ-ДД"},
                "road": {"type": "string",
                         "description": f"код дороги выгрузки, один из: {', '.join(ROADS)}"},
                "station": {"type": "string",
                            "description": "название станции выгрузки, как в справочнике"},
                "rod": {"type": "string",
                        "description": f"род подвижного состава: {', '.join(RODS)}"},
                "group_by": {"type": "string", "enum": ["day", "month"],
                             "description": "разбить результат по дням или по месяцам"},
            },
            "required": ["date_from", "date_to"],
        },
    },
}


In [40]:
# ═══════════════════════════ TODO 1 из 4 ═══════════════════════════
# Что здесь написать: описания двух оставшихся инструментов и общий список TOOLS.
#
# 1) TOOL_QUERY_LOAD — по образцу TOOL_QUERY_UNLOAD выше, но про ПОГРУЗКУ.
#    Функция tool_query_load принимает те же аргументы: date_from, date_to,
#    road, station, rod, group_by. Обязательные — обе даты.
#    В описании скажите явно, что это погрузка, а не выгрузка: два похожих
#    инструмента модель путает чаще всего, и лечится это одной фразой.
#
# 2) TOOL_FIND_WAGONS — про текущую дислокацию. Аргументы функции
#    tool_find_wagons: wagon_numbers (массив строк, до 20 номеров), station, road.
#    Обязательных аргументов нет, но нужен хотя бы один — напишите это в описании.
#    Массив в JSON-схеме описывается так:
#        "wagon_numbers": {"type": "array", "items": {"type": "string"},
#                          "description": "..."}
#    И обязательно скажите, что это СРЕЗ НА СЕГОДНЯ, истории перемещений в нём нет.
#    Иначе модель будет отвечать этим инструментом на вопросы «где стоял в феврале».
#
# 3) TOOLS = [TOOL_QUERY_UNLOAD, TOOL_QUERY_LOAD, TOOL_FIND_WAGONS]
#
# check() смотрит: три инструмента, имена совпадают с именами функций,
# описание каждого не короче 40 символов, схема аргументов — объект с properties.
# ═══════════════════════════════════════════════════════════════════


TOOL_QUERY_LOAD = {
    "type": "function",
    "function": {
        "name": "query_load",
        "description": (
            "Возвращает статистику ПОГРУЗКИ (не выгрузки!) вагонов за период "
            "по данным витрины загрузок. Используй этот инструмент для вопросов "
            "о том, сколько груза отправлено/погружено, в отличие от выгрузки, "
            "которая про то, сколько прибыло и было разгружено."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "date_from": {"type": "string", "format": "date",
                               "description": "начало периода (включительно), формат YYYY-MM-DD"},
                "date_to": {"type": "string", "format": "date",
                             "description": "конец периода (включительно), формат YYYY-MM-DD"},
                "road": {"type": "string", "description": "название дороги, опционально"},
                "station": {"type": "string", "description": "название станции, опционально"},
                "rod": {"type": "string", "description": "род вагона (например ПВ, ЦС), опционально"},
                "group_by": {"type": "string", "description": "по какому полю группировать результат, опционально"},
            },
            "required": ["date_from", "date_to"],
        },
    },
}

TOOL_FIND_WAGONS = {
    "type": "function",
    "function": {
        "name": "find_wagons",
        "description": (
            "Возвращает текущую дислокацию вагонов — СРЕЗ НА СЕГОДНЯ, истории "
            "перемещений в нём нет. Не используй этот инструмент для вопросов "
            "о том, где вагон находился в прошлом (например, «в феврале»). "
            "Нужен хотя бы один из аргументов: номера вагонов, станция или дорога."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "wagon_numbers": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "список номеров вагонов, до 20 штук, опционально",
                },
                "station": {"type": "string", "description": "станция текущего местонахождения, опционально"},
                "road": {"type": "string", "description": "дорога текущего местонахождения, опционально"},
            },
            "required": [],
        },
    },
}


TOOL_QUERY_GU12 = {
    "type": "function",
    "function": {
        "name": "query_gu12",
        "description": (
            "Среднесуточное количество вагонов в заявках ГУ-12 за период. "
            "ВАЖНО: это снимок остатка на каждый день (запас), а не поток вагонов, "
            "поэтому за диапазон дат считается СРЕДНЕЕ по дням, а не сумма — "
            "суммировать снимки за разные дни бессмысленно, это завышает уровень. "
            "Можно сузить по дороге, станции и роду подвижного состава."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "date_from": {"type": "string", "description": "начало периода, ГГГГ-ММ-ДД"},
                "date_to": {"type": "string", "description": "конец периода включительно, ГГГГ-ММ-ДД"},
                "road": {"type": "string", "description": f"код дороги, один из: {', '.join(ROADS)}"},
                "station": {"type": "string", "description": "название станции, как в справочнике"},
                "rod": {"type": "string", "description": f"род подвижного состава: {', '.join(RODS)}"},
            },
            "required": ["date_from", "date_to"],
        },
    },
}

TOOLS = [TOOL_QUERY_UNLOAD, TOOL_QUERY_LOAD, TOOL_FIND_WAGONS, TOOL_QUERY_GU12]


# ── Ниже менять не нужно: быстрая проверка формата ──
if not TOOLS:
    print("⛔ TOOLS пуст — TODO 1 ещё не сделан.")
else:
    for t in TOOLS:
        f = t.get("function", {})
        print(f"{f.get('name', '???'):<14} аргументов: "
              f"{len(f.get('parameters', {}).get('properties', {})):>2}  "
              f"описание: {len(f.get('description', ''))} симв.")


query_unload   аргументов:  6  описание: 212 симв.
query_load     аргументов:  6  описание: 247 симв.
find_wagons    аргументов:  3  описание: 261 симв.
query_gu12     аргументов:  5  описание: 322 симв.


---
## 7. TODO 2 — диспетчер: выполнить то, что попросила модель

Модель вернула имя и аргументы. Дальше работает ваш код: найти функцию по имени,
вызвать, вернуть результат строкой.

Отдельно про ошибки. Инструмент может упасть — пользователь спросил про дорогу, которой
нет, или забыл период. **Исключение наружу выпускать нельзя:** упавший процесс не даёт
модели шанса исправиться. Ошибку надо вернуть **как результат инструмента**, текстом.
Тогда модель прочитает «дороги ВСТ нет, есть такие-то» и либо переспросит, либо исправится.


In [41]:
# Готово: связь между именем для модели и функцией на Python.
TOOL_FUNCTIONS = {
    "query_unload": tool_query_unload,
    "query_load": tool_query_load,
    "find_wagons": tool_find_wagons,
    "query_gu12": tool_query_gu12
}


In [42]:
# ═══════════════════════════ TODO 2 из 4 ═══════════════════════════
# Что здесь написать: диспетчер вызовов.
#
# Сигнатура:  call_tool(name: str, arguments: dict) -> str
#
# Четыре шага:
#   1) найти функцию в TOOL_FUNCTIONS по имени name;
#      не нашлась -> вернуть текст об этом (json.dumps словаря с ключом "error"),
#      и перечислить доступные имена: модель ошиблась именем, пусть исправится;
#   2) вызвать функцию: fn(**arguments);
#   3) любое исключение поймать и вернуть тем же способом — словарём с "error"
#      и текстом ошибки. Никаких raise наружу: цикл агента не должен падать;
#   4) успешный результат — это словарь; верните json.dumps(result,
#      ensure_ascii=False). Модель ждёт строку, а не объект Python.
#
# ensure_ascii=False обязателен: иначе русские названия станций уедут
# в ДОЛ... и модель будет сравнивать их с тем, что написал
# пользователь, вслепую.
# ═══════════════════════════════════════════════════════════════════


def call_tool(name: str, arguments: dict) -> str:
    fn = TOOL_FUNCTIONS.get(name)
    if fn is None:
        return json.dumps({
            "error": f"Неизвестный инструмент: {name!r}. "
                     f"Доступные инструменты: {list(TOOL_FUNCTIONS)}"
        }, ensure_ascii=False)

    try:
        result = fn(**arguments)
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"}, ensure_ascii=False)

    return json.dumps(result, ensure_ascii=False)


# ── Ниже менять не нужно: три проверки диспетчера ──
try:
    ok = call_tool("query_unload", {"date_from": "2021-03-01", "date_to": "2021-03-31",
                                    "road": "ЛСН"})
    bad_road = call_tool("query_unload", {"date_from": "2021-03-01", "date_to": "2021-03-31",
                                          "road": "ВСТ"})
    bad_name = call_tool("query_everything", {})
    need(ok, 2, "call_tool ничего не вернул")
    print("нормальный вызов :", ok)
    print("плохая дорога    :", bad_road)
    print("нет такого имени :", bad_name)
    print("\nВсе три строки — строки, а не исключения. Это то, что нужно.")
except TodoNotDone as e:
    explain_todo(e)


нормальный вызов : {"metric": "выгружено вагонов", "period": ["2021-03-01", "2021-03-31"], "wagons": 9761, "filters": {"road": "ЛСН"}}
плохая дорога    : {"error": "ValueError: Дорога: значения 'ВСТ' в наборе нет. Допустимые значения: ГРН, ДОЛ, ЗАЛ, КРЖ, ЛСН, ОЗР, ПРБ, ПУЩ, СТП, ТУН."}
нет такого имени : {"error": "Неизвестный инструмент: 'query_everything'. Доступные инструменты: ['query_unload', 'query_load', 'find_wagons', 'query_gu12']"}

Все три строки — строки, а не исключения. Это то, что нужно.


---
## 8. TODO 3 — цикл агента

Теперь собираем всё вместе. Цикл — это пять строк смысла:

1. отправить модели переписку и список инструментов;
2. вернулся текст — всё, это ответ, выходим;
3. вернулись вызовы — выполнить каждый через `call_tool`;
4. дописать в переписку и ответ модели, и результаты инструментов;
5. повторить — но **не больше `max_steps` раз**.

Пятый пункт — не украшение. Без него модель, которая не может получить нужное,
будет звать инструмент по кругу, пока не кончатся деньги на ключе.

Четыре помощника ниже готовы: они прячут возню с форматом сообщений SDK, чтобы вы писали
логику, а не подбирали ключи словарей.


In [43]:
def assistant_message(msg) -> dict:
    """Ответ модели -> словарь для переписки. Готовый код."""
    out = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        out["tool_calls"] = [
            {"id": c.id, "type": "function",
             "function": {"name": c.function.name, "arguments": c.function.arguments}}
            for c in msg.tool_calls
        ]
    return out


def parse_arguments(tool_call) -> dict:
    """Аргументы приходят строкой JSON. Готовый код."""
    try:
        return json.loads(tool_call.function.arguments or "{}")
    except json.JSONDecodeError:
        return {}


def log_step(step: int, name: str, args: dict, result: str) -> None:
    """Печать одной итерации. Готовый код."""
    short = result if len(result) <= 160 else result[:157] + "…"
    print(f"  шаг {step}: {name}({json.dumps(args, ensure_ascii=False)})")
    print(f"          -> {short}")


def start_messages(system: str | None, question: str) -> list[dict]:
    """Начало переписки: системное сообщение, если оно есть, и вопрос. Готовый код."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": question})
    return messages


In [44]:
# ═══════════════════════════ TODO 3 из 4 ═══════════════════════════
# Что здесь написать: цикл агента.
#
# Сигнатура:
#   run_agent(question, system=None, max_steps=5, model=MODEL, verbose=True) -> dict
#
# Возвращает словарь с четырьмя ключами — их читает check():
#   {"answer": текст ответа, "steps": сколько итераций сделали,
#    "calls":  список вызовов [{"name":..., "arguments":..., "result":...}, ...],
#    "stopped_by": "text" если модель ответила текстом, "max_steps" если упёрлись
#                  в ограничитель}
#
# Как писать:
#   1) messages = start_messages(system, question) — этот хелпер уже готов;
#   2) цикл for step in range(1, max_steps + 1):
#        r = client.chat.completions.create(model=model, messages=messages,
#                                           tools=TOOLS, tool_choice="auto",
#                                           temperature=0)
#        msg = r.choices[0].message
#   3) если msg.tool_calls пуст — модель ответила текстом: вернуть словарь
#      со stopped_by="text";
#   4) иначе: messages.append(assistant_message(msg)) — ответ модели обязан
#      попасть в переписку ДО результатов, иначе провайдер вернёт ошибку формата;
#   5) для каждого вызова c в msg.tool_calls:
#        args = parse_arguments(c)
#        result = call_tool(c.function.name, args)
#        записать в calls, при verbose — log_step(step, c.function.name, args, result)
#        messages.append({"role": "tool", "tool_call_id": c.id, "content": result})
#   6) цикл кончился, а модель всё зовёт инструменты — вернуть словарь
#      со stopped_by="max_steps" и тем ответом, что есть (можно пустым).
#
# Если забыть tool_call_id в пункте 5 — провайдер ответит ошибкой 400.
# Это нормально: прочитайте её текст, он говорит ровно об этом.
# ═══════════════════════════════════════════════════════════════════


def run_agent(question: str, system: str | None = None, max_steps: int = 5,
              model: str = MODEL, verbose: bool = True) -> dict:
    messages = start_messages(system, question)
    calls = []

    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0,
        )
        msg = r.choices[0].message

        if not msg.tool_calls:
            return {
                "answer": msg.content or "",
                "steps": step,
                "calls": calls,
                "stopped_by": "text",
            }

        messages.append(assistant_message(msg))

        for c in msg.tool_calls:
            args = parse_arguments(c)
            result = call_tool(c.function.name, args)
            calls.append({"name": c.function.name, "arguments": args, "result": result})
            if verbose:
                log_step(step, c.function.name, args, result)
            messages.append({"role": "tool", "tool_call_id": c.id, "content": result})

    return {
        "answer": msg.content or "",
        "steps": max_steps,
        "calls": calls,
        "stopped_by": "max_steps",
    }

# ── Ниже менять не нужно: первый живой прогон, пока без системного промпта ──
try:
    demo = need(run_agent("Сколько вагонов выгрузили на дороге ЛСН в марте 2021 года?"),
                3, "run_agent ничего не вернул")
    print("\nответ:", demo["answer"])
    print(f"итераций: {demo['steps']}, остановились по: {demo['stopped_by']}")
except TodoNotDone as e:
    explain_todo(e)


  шаг 1: query_unload({"date_from": "2021-03-01", "date_to": "2021-03-31", "road": "ЛСН"})
          -> {"metric": "выгружено вагонов", "period": ["2021-03-01", "2021-03-31"], "wagons": 9761, "filters": {"road": "ЛСН"}}

ответ: За март 2021 года на дороге **ЛСН** выгружено **9 761 вагон**.
итераций: 2, остановились по: text


---
## 9. Зачем нужен ограничитель

Ячейка готова. Один и тот же вопрос задаётся дважды: с `max_steps=1` и с `max_steps=5`.

Смотрите на строку `остановка`. При `max_steps=1` модель успевает только **попросить**
вызов инструмента — ответа не будет, цикл закончится не потому, что агент справился,
а потому, что его остановили. При `max_steps=5` тот же вопрос доходит до ответа за две
итерации.

Отсюда правило: **ограничитель решает, когда цикл закончится, если модель не сходится.**
Без него сходиться она может бесконечно — и каждая итерация стоит денег с вашего ключа.


In [45]:
limiter_question = "Сколько вагонов выгрузили на дороге ЛСН в марте 2021 года?"

try:
    for limit in (1, 5):
        r = need(run_agent(limiter_question, max_steps=limit, verbose=False),
                 3, "run_agent ничего не вернул")
        answer = r["answer"].strip() or "— ответа нет —"
        print(f"max_steps={limit}: итераций {r['steps']}, вызовов {len(r['calls'])}, "
              f"остановка по «{r['stopped_by']}»")
        print(f"            ответ: {answer[:120]}\n")
    print("Один и тот же вопрос. Разница только в том, сколько итераций вы разрешили.")
except TodoNotDone as e:
    explain_todo(e)


max_steps=1: итераций 1, вызовов 1, остановка по «max_steps»
            ответ: — ответа нет —

max_steps=5: итераций 2, вызовов 1, остановка по «text»
            ответ: За март 2021 года на дороге **ЛСН** выгружено **9 761 вагон**.

Один и тот же вопрос. Разница только в том, сколько итераций вы разрешили.


---
## 10. TODO 4 — границы в системном промпте

Агент уже работает. Проблема в том, что он **уверен всегда** — даже когда не должен.

Спросите его «сколько вагонов выгрузили на дороге ЛСН?» без периода: он не переспросит,
а подставит период сам, молча. На практике 1 у нас был флаг `is_ambiguous` — модель
помечала неуверенность. Агент умеет больше: он может **задать вопрос**.

Границы, которые нужно проговорить, — из технического задания на агента
(`data/STUDENT_TASK.md`, пункты 1, 2 и 4).


In [61]:
# ═══════════════════════════ TODO 4 из 4 ═══════════════════════════
# Что здесь написать: системный промпт с границами.
#
# Сигнатура:  build_system_prompt(today: str = TODAY) -> str
#
# Что обязательно должно быть в тексте (это проверяет check() поведением агента):
#   1) сегодняшняя дата и период данных: подставьте today, а не литерал —
#      у модели нет часов, и «март» без года она поймёт как придётся;
#   2) ПРАВИЛО ПЕРИОДА: если вопросу нужен период, а пользователь его не назвал —
#      НЕ вызывать инструмент и НЕ подставлять период самому, а задать
#      уточняющий вопрос. Это п. 1 STUDENT_TASK;
#   3) ПРАВИЛО ФИЛЬТРОВ: не применять фильтры, которых пользователь не называл
#      (дорога, станция, род вагона). Это п. 2 STUDENT_TASK;
#   4) ПРАВИЛО ИСТОЧНИКА: find_wagons — это срез на сегодня. На вопрос о том,
#      где вагон был раньше, надо сказать, что таких данных нет, а не выдавать
#      текущее положение за историческое. Это п. 4 STUDENT_TASK;
#   5) в финальном ответе называть период, применённые фильтры и единицы
#      (вагоны), а внутренние имена колонок и инструментов пользователю
#      не показывать.
#
# Пишите по-русски, обычным текстом, 10–20 строк. Это не заклинание:
# формулируйте так, как объяснили бы новому сотруднику.
# ═══════════════════════════════════════════════════════════════════


def build_system_prompt(today: str = TODAY) -> str:
    return f"""Ты агент, который отвечает на вопросы про вагонные перевозки,
используя инструменты query_unload, query_load и find_wagons.

Сегодняшняя дата: {today}. Данные в системе покрывают период
ноябрь 2020 — апрель 2021 — учитывай это, когда пользователь называет период.

ПРАВИЛО ПЕРИОДА: если для ответа на вопрос нужен период (диапазон дат), а
пользователь его не назвал явно (не указал год, не указал ни один из месяцев
или границ) — не вызывай инструмент и не подставляй период сам. Вместо этого
задай пользователю уточняющий вопрос, за какой именно период нужны данные.

ПРАВИЛО ФИЛЬТРОВ: не применяй фильтры (дорога, станция, род вагона), которые
пользователь явно не назвал. Если фильтр не упомянут — вызывай инструмент без
него, не додумывай значение за пользователя.

ПРАВИЛО ИСТОЧНИКА: find_wagons отдаёт срез дислокации только на сегодняшний
день, истории перемещений в нём нет. Если пользователь спрашивает, где вагон
находился раньше (например, в прошлом месяце) — не вызывай find_wagons и не
выдавай текущее положение за историческое, а честно скажи, что таких
исторических данных нет.

ПРАВИЛО ГУ-12: query_gu12 возвращает СРЕДНЕЕ количество вагонов в заявках за период,
потому что данные ГУ-12 — это ежедневный снимок остатка, а не поток вагонов. Никогда
не складывай значения GU12_WAG_CNT за разные дни самостоятельно и не проси у
пользователя "сумму" по этому источнику — только среднесуточное значение, которое
и так считает инструмент.

ВАЖНО: query_gu12 не умеет разбивать результат по дням или месяцам (у него нет
group_by). Если пользователь просит такую детализацию по заявкам ГУ-12 — прямо
скажи, что этот инструмент даёт только одно среднее число за весь период, и
предложи либо вызвать его отдельно на каждый интересующий подпериод, либо
уточнить, устроит ли общее среднее.

В финальном ответе называй период, за который считал, и фильтры, которые
применил (или явно скажи, что фильтров не было), а также единицы измерения
(вагоны). Не показывай пользователю внутренние имена инструментов, функций
или колонок данных — только человеческий язык."""

# ── Ниже менять не нужно: сравнение с границами и без ──
try:
    SYSTEM = need(build_system_prompt(), 4, "build_system_prompt вернул пустой промпт")
    question = "Сколько вагонов выгрузили на дороге ЛСН?"

    print("─── без границ ───")
    without = need(run_agent(question, verbose=False), 3, "run_agent ничего не вернул")
    print(without["answer"][:300])

    print("\n─── с границами ───")
    with_rules = need(run_agent(question, system=SYSTEM, verbose=False), 3,
                      "run_agent ничего не вернул")
    print(with_rules["answer"][:300])

    print(f"\nвызовов инструментов: без границ — {len(without['calls'])}, "
          f"с границами — {len(with_rules['calls'])}")
except TodoNotDone as e:
    explain_todo(e)


─── без границ ───
За 2025 год (01.01–31.12.2025) на дороге **ЛСН** выгружено **0 вагонов**.

Обратите внимание: инструмент требует обязательный период, а вы его не указали — я взял текущий год целиком. Если нужен другой интервал (например, конкретный месяц или квартал), уточните даты, и я пересчитаю. Также стоит пров

─── с границами ───
Уточните, пожалуйста, за какой период нужны данные о выгрузке на дороге ЛСН (например, конкретный месяц или диапазон дат).

вызовов инструментов: без границ — 1, с границами — 0


---
## 11. Прогон четырёх сценариев

Ячейка готова. Четыре вопроса, по одному на каждое поведение, которое мы разбирали:

| Вопрос | Что проверяем |
|---|---|
| выгрузка ЛСН за март 2021 | обычный расчёт: нужный инструмент, верное число |
| выгрузка ЛСН без периода | агент переспрашивает, а не выдумывает период |
| выгрузка по станции, которой нет в справочнике | ошибка инструмента не роняет цикл |
| где стоял вагон в феврале | агент ограничивает источник, а не выдаёт срез за историю |

Выполняйте её после всех четырёх TODO. Это 4 живых диалога, примерно полминуты.


In [47]:
SCENARIOS = {
    "период_есть": "Сколько вагонов выгрузили на дороге ЛСН в марте 2021 года?",
    "период_не_указан": "Сколько вагонов выгрузили на дороге ЛСН?",
    "ошибка_инструмента": "Сколько вагонов выгрузили на станции ПЕТУШКИ в марте 2021 года?",
    "история_дислокации": "Где стоял вагон 83196971 в феврале 2021 года?",
}

RESULTS: dict[str, dict] = {}
try:
    SYSTEM = need(build_system_prompt(), 4, "build_system_prompt вернул пустой промпт")
    for key, q in SCENARIOS.items():
        print(f"\n=== {key} ===\n{q}")
        RESULTS[key] = need(run_agent(q, system=SYSTEM, verbose=False), 3,
                            "run_agent ничего не вернул")
        r = RESULTS[key]
        print(f"  вызовов: {len(r['calls'])}, итераций: {r['steps']}, "
              f"остановка: {r['stopped_by']}")
        print(f"  ответ: {r['answer'][:220]}")
except TodoNotDone as e:
    explain_todo(e)



=== период_есть ===
Сколько вагонов выгрузили на дороге ЛСН в марте 2021 года?
  вызовов: 1, итераций: 2, остановка: text
  ответ: За март 2021 года (с 1 по 31 марта) на дороге ЛСН было выгружено **9 761 вагон**.

Фильтры: только дорога выгрузки — ЛСН. Единица измерения — вагоны.

=== период_не_указан ===
Сколько вагонов выгрузили на дороге ЛСН?
  вызовов: 0, итераций: 1, остановка: text
  ответ: Уточните, пожалуйста, за какой период нужны данные о выгрузке на дороге ЛСН? (например, конкретный месяц или диапазон дат)

=== ошибка_инструмента ===
Сколько вагонов выгрузили на станции ПЕТУШКИ в марте 2021 года?
  вызовов: 1, итераций: 2, остановка: text
  ответ: Не удалось получить данные: станции с названием «ПЕТУШКИ» в справочнике нет.

Уточните, пожалуйста, точное название станции (как в справочнике) — и я повторю запрос за март 2021 года.

=== история_дислокации ===
Где стоял вагон 83196971 в феврале 2021 года?
  вызовов: 0, итераций: 1, остановка: text
  ответ: Такой информации у мен

---
## 12. `check()` — критерий сдачи

Шесть проверок. Работа сдана при **6/6**.

Проверки смотрят на **поведение вашего агента**, а не на текст промпта: какой инструмент
он позвал, с какими аргументами, довёл ли цикл до конца и что сделал там, где данных нет.


In [63]:
def _calls_named(result: dict, name: str) -> list[dict]:
    return [c for c in result.get("calls", []) if c["name"] == name]


def _result_values(result: dict, name: str, key: str) -> list:
    out = []
    for c in _calls_named(result, name):
        try:
            payload = json.loads(c["result"])
        except (json.JSONDecodeError, TypeError):
            continue
        if key in payload:
            out.append(payload[key])
    return out


def _need_results():
    if not RESULTS:
        raise AssertionError("нет прогона: выполните ячейку «Прогон четырёх сценариев» выше")


# ── сами проверки: каждая возвращает (успех, сообщение) ─────────────────────

def _check_tools_declared():
    if not TOOLS:
        return False, "TOOLS пуст — TODO 1 не сделан"
    names = [t.get("function", {}).get("name") for t in TOOLS]
    if sorted(n for n in names if n) != sorted(TOOL_FUNCTIONS):
        return False, f"имена инструментов {names} не совпадают с {sorted(TOOL_FUNCTIONS)}"
    for t in TOOLS:
        f = t.get("function", {})
        if len(f.get("description", "")) < 40:
            return False, f"описание '{f.get('name')}' короче 40 символов — модель его не поймёт"
        params = f.get("parameters", {})
        if params.get("type") != "object" or not isinstance(params.get("properties"), dict):
            return False, f"схема аргументов '{f.get('name')}' — не объект с properties"
    req = TOOLS[[t["function"]["name"] for t in TOOLS].index("query_load")]
    if sorted(req["function"]["parameters"].get("required", [])) != ["date_from", "date_to"]:
        return False, "у query_load обязательными должны быть ровно date_from и date_to"
    return True, f"три инструмента объявлены: {', '.join(sorted(n for n in names if n))}"


def _check_number():
    _need_results()
    r = RESULTS["период_есть"]
    wagons = _result_values(r, "query_unload", "wagons")
    expected = FACTS["unload_lsn_2021_03"]
    if not wagons:
        return False, "агент не вызвал query_unload на вопросе с явным периодом"
    if expected not in wagons:
        return False, (f"инструмент вернул {wagons}, а по данным за март 2021 на ЛСН "
                       f"{expected}: проверьте аргументы, которые собрала модель")
    flat = "".join(ch for ch in r["answer"] if not ch.isspace())
    if str(expected) not in flat:
        return False, f"число {expected} посчитано, но в ответе пользователю его нет"
    return True, f"верное число {expected} посчитано инструментом и попало в ответ"


def _check_loop_finished():
    """Цикл обязан заканчиваться всегда: либо текстом, либо ограничителем.

    Проверяем код студента, а не упорство модели: слабая модель может так и не
    прочитать текст ошибки инструмента — это законный выход по max_steps.
    """
    _need_results()
    for key, r in RESULTS.items():
        if r.get("stopped_by") not in ("text", "max_steps"):
            return False, f"сценарий '{key}': run_agent не вернул stopped_by"
        if r.get("steps", 99) > 5:
            return False, f"сценарий '{key}': {r['steps']} итераций при max_steps=5"
    main = RESULTS["период_есть"]
    if main["stopped_by"] != "text":
        return False, ("на вопросе с явным периодом агент упёрся в ограничитель "
                       "вместо ответа — проверьте описания инструментов")
    steps = {k: r["steps"] for k, r in RESULTS.items()}
    return True, f"все четыре диалога завершились, итерации: {steps}"


def _check_asks_for_period():
    _need_results()
    r = RESULTS["период_не_указан"]
    if _calls_named(r, "query_unload") or _calls_named(r, "query_load"):
        return False, ("на вопрос без периода агент всё равно посчитал: "
                       "правило про уточнение не сработало")
    low = r["answer"].lower()
    asks = "?" in r["answer"] or any(
        m in low for m in ("уточн", "какой период", "каком периоде", "какие даты",
                           "укажите", "назовите", "за какой", "с какой по какую"))
    if not asks:
        return False, ("агент не посчитал, но и не переспросил про период: "
                       f"ответ был «{r['answer'][:120]}»")
    return True, "на вопрос без периода агент переспросил, а не выдумал период"


def _check_survives_tool_error():
    """Ошибка инструмента должна возвращаться строкой, а не исключением.

    Диспетчер проверяем напрямую: позовёт модель ошибочный инструмент или
    догадается заранее — это её дело, а не оценка вашего кода.
    """
    _need_results()
    try:
        probe = call_tool("query_unload", {"date_from": "2021-03-01",
                                           "date_to": "2021-03-31", "road": "ВСТ"})
    except Exception as e:  # noqa: BLE001
        return False, (f"call_tool выпустил исключение наружу ({type(e).__name__}) — "
                       "ошибку инструмента надо возвращать как результат")
    try:
        payload = json.loads(probe)
    except (json.JSONDecodeError, TypeError):
        return False, f"call_tool вернул не JSON-строку: {str(probe)[:80]}"
    if "error" not in payload:
        return False, f"call_tool не сообщил об ошибке инструмента: {probe[:120]}"

    r = RESULTS["ошибка_инструмента"]
    if r["stopped_by"] not in ("text", "max_steps"):
        return False, "на сценарии с ошибкой цикл не завершился штатно"

    flowed = [c for c in r.get("calls", []) if '"error"' in c["result"]]
    tail = ("ошибка прошла через цикл и агент ответил" if flowed
            else "в этом прогоне модель обошлась без ошибочного вызова")
    return True, f"диспетчер возвращает ошибку строкой; {tail}"


def _check_limits_source():
    _need_results()
    r = RESULTS["история_дислокации"]
    low = r["answer"].lower()
    markers = ("нет", "не хран", "не содерж", "текущ", "только на", "истор", "невозможно")
    if not any(m in low for m in markers):
        return False, ("агент не сказал, что истории перемещений в данных нет: "
                       f"ответ был «{r['answer'][:120]}»")
    return True, "агент ограничил источник, а не выдал текущий срез за историю"


CHECKS = [
    ("1. три инструмента объявлены", _check_tools_declared),
    ("2. верное число на вопросе с периодом", _check_number),
    ("3. цикл завершается и укладывается в max_steps", _check_loop_finished),
    ("4. вопрос без периода — уточнение", _check_asks_for_period),
    ("5. ошибка инструмента не роняет цикл", _check_survives_tool_error),
    ("6. исторический вопрос — ограничение источника", _check_limits_source),
]


def check() -> int:
    passed = 0
    for title, fn in CHECKS:
        try:
            ok, message = fn()
        except AssertionError as e:
            ok, message = False, str(e)
        except Exception as e:  # noqa: BLE001
            ok, message = False, f"{type(e).__name__}: {e}"
        print(f"{'✅' if ok else '❌'} {title}\n    {message}")
        passed += bool(ok)
    print(f"\nИтого: {passed} из {len(CHECKS)}")
    if passed == len(CHECKS):
        print("Работа сдана. Сохраните вывод этой ячейки в PR.")
    return passed


check()


✅ 1. три инструмента объявлены
    три инструмента объявлены: find_wagons, query_gu12, query_load, query_unload
✅ 2. верное число на вопросе с периодом
    верное число 9761 посчитано инструментом и попало в ответ
✅ 3. цикл завершается и укладывается в max_steps
    все четыре диалога завершились, итерации: {'период_есть': 2, 'период_не_указан': 1, 'ошибка_инструмента': 2, 'история_дислокации': 1}
✅ 4. вопрос без периода — уточнение
    на вопрос без периода агент переспросил, а не выдумал период
✅ 5. ошибка инструмента не роняет цикл
    диспетчер возвращает ошибку строкой; ошибка прошла через цикл и агент ответил
✅ 6. исторический вопрос — ограничение источника
    агент ограничил источник, а не выдал текущий срез за историю

Итого: 6 из 6
Работа сдана. Сохраните вывод этой ячейки в PR.


6

---
## 13. Домашнее задание

1. **Довести `check()` до 6/6.** Правьте описания инструментов и системный промпт —
   готовые функции и сам `check()` менять нельзя.

2. **Добавить четвёртый инструмент — заявки ГУ-12.** Витрина `gu12` — это **суточные
   снимки запаса заявок**: строки за разные дни складывать нельзя, получится бессмыслица
   (п. 5 `data/STUDENT_TASK.md`). Инструмент должен считать **среднесуточное** количество
   вагонов в заявках за период, а его описание — объяснять модели, почему не сумму.
   Данные: `V_DM_GU12.csv` из `data/dataset_student.zip`, колонка `GU12_WAG_CNT`.

3. **Три строки текстом:** на каком вопросе ваш агент ошибся и что вы поменяли —
   описание инструмента или системный промпт. Это самое полезное наблюдение вечера,
   и на защите его спросят.

**Контрольная точка на занятии 3 (17 сентября): «Заявка на проект».** Команда 2–3 человека,
пара осей из таблицы в `syllabus.md`, домен и то, что считается результатом. Комбинации
не повторяются между командами и фиксируются по порядку заявок.


Агент ошибся на вопросе: Покажи среднесуточное количество по дням за первую неделю марта. Вместо разбивки по дням он молча выдал одно среднее число за всю неделю, не предупредив, что детализация недоступна. Я добавил в системный промпт правило: если у query_gu12 нет group_by, а пользователь просит разбивку, агент должен честно сказать об этом ограничении, а не подменять вопрос агрегатом. После правки агент стал прямо объяснять ограничение и предлагать пользователю выбрать: общее среднее или семь отдельных вызовов по дням.

## Промты для нахождения ошибки

In [51]:
result = run_agent(
    "Сколько всего вагонов было в заявках ГУ-12 за март и апрель вместе?",
    system=SYSTEM,
    verbose=True,
)
print("\nответ агента:", result["answer"])


result = run_agent(
    "Сколько в среднем вагонов в заявках ГУ-12 на дороге ЛСН?",
    system=SYSTEM,
    verbose=True,
)
print("\nответ агента:", result["answer"])

result = run_agent(
    "Сколько погрузили на дороге ЛСН в марте 2021?",
    system=SYSTEM,
    verbose=True,
)
print("\nответ агента:", result["answer"])


result = run_agent(
    "Сколько вагонов в заявках на дороге ВСТ за март?",
    system=SYSTEM,
    verbose=True,
)
print("\nответ агента:", result["answer"])


ответ агента: Правильно уточню: по заявкам ГУ-12 данные — это ежедневный снимок остатка, поэтому за период считается **среднесуточное** количество вагонов, а не сумма. Складывать значения за март и апрель (и тем более за отдельные дни) нельзя — это завысит результат.

Поэтому я могу дать среднесуточное количество вагонов в заявках за период с 1 марта по 30 апреля 2021. Уточните, пожалуйста: нужны ли какие-то фильтры (дорога, станция, род подвижного состава) или считать в целом по всем?

ответ агента: Уточните, пожалуйста, за какой период нужны данные по заявкам ГУ-12 на дороге ЛСН?
  шаг 1: query_load({"date_from": "2021-03-01", "date_to": "2021-03-31", "road": "ЛСН"})
          -> {"metric": "погружено вагонов", "period": ["2021-03-01", "2021-03-31"], "wagons": 11091, "filters": {"road": "ЛСН"}}

ответ агента: За март 2021 года на дороге ЛСН погружено **11 091 вагон**.

Период: 1–31 марта 2021. Фильтр: дорога ЛСН (род подвижного состава и станция не задавались).

ответ агента: Не мог

In [52]:
result = run_agent(
    "Сколько в среднем вагонов было в заявках ГУ-12 за март и апрель на дороге ЛСН?",
    system=SYSTEM,
    verbose=True,
)
print("\nответ агента:", result["answer"])

  шаг 1: query_gu12({"date_from": "2021-03-01", "date_to": "2021-04-30", "road": "ЛСН"})
          -> {"metric": "среднесуточное количество вагонов в заявках ГУ-12", "period": ["2021-03-01", "2021-04-30"], "days_covered": 61, "wagons_avg_per_day": 1297.8, "fi…

ответ агента: За период с 1 марта по 30 апреля 2021 года среднесуточное количество вагонов в заявках ГУ-12 на дороге ЛСН составило **1297,8 вагона в сутки**.

Фильтр: дорога ЛСН. Это именно среднесуточное значение (снимок остатка заявок), а не сумма за период — суммировать данные по дням здесь нельзя.


In [53]:
result = run_agent(
    "Сколько в среднем вагонов было в заявках ГУ-12 отдельно в марте и отдельно в апреле?",
    system=SYSTEM,
    verbose=True,
)
print("\nответ агента:", result["answer"])

  шаг 1: query_gu12({"date_from": "2021-03-01", "date_to": "2021-03-31"})
          -> {"metric": "среднесуточное количество вагонов в заявках ГУ-12", "period": ["2021-03-01", "2021-03-31"], "days_covered": 31, "wagons_avg_per_day": 8151.9, "fi…
  шаг 1: query_gu12({"date_from": "2021-04-01", "date_to": "2021-04-30"})
          -> {"metric": "среднесуточное количество вагонов в заявках ГУ-12", "period": ["2021-04-01", "2021-04-30"], "days_covered": 30, "wagons_avg_per_day": 9071.8, "fi…

ответ агента: Вот данные по заявкам ГУ-12 (среднесуточное количество вагонов, фильтры не применялись):

- **Март 2021** (01.03–31.03): в среднем **8 151,9 вагона** в сутки.
- **Апрель 2021** (01.04–30.04): в среднем **9 071,8 вагона** в сутки.

Обратите внимание: это именно среднесуточный остаток вагонов в заявках, а не суммарный поток за месяц — складывать значения по дням здесь нельзя.


In [60]:
result = run_agent(
    "Покажи среднесуточное количество вагонов в заявках ГУ-12 по дням за первую неделю марта 2021 на дороге ЛСН",
    system=SYSTEM,
    verbose=True,
)
print("\nответ агента:", result["answer"])

  шаг 1: query_gu12({"date_from": "2021-03-01", "date_to": "2021-03-07", "road": "ЛСН"})
          -> {"metric": "среднесуточное количество вагонов в заявках ГУ-12", "period": ["2021-03-01", "2021-03-07"], "days_covered": 7, "wagons_avg_per_day": 1232.0, "fil…

ответ агента: За период с 1 по 7 марта 2021 года (первая неделя марта) по дороге ЛСН среднесуточное количество вагонов в заявках ГУ-12 составило **1232 вагона в сутки**.

Применён фильтр: дорога ЛСН. Других фильтров (станция, род подвижного состава) не применялось.

Обратите внимание: это именно среднесуточное значение — заявки ГУ-12 представляют собой снимок остатка на каждый день, а не поток вагонов, поэтому суммировать значения по дням было бы некорректно.


In [62]:
result = run_agent(
    "Покажи среднесуточное количество вагонов в заявках ГУ-12 по дням за первую неделю марта 2021 на дороге ЛСН",
    system=SYSTEM,
    verbose=True,
)
print("\nответ агента:", result["answer"])


ответ агента: Инструмент для заявок ГУ-12 не умеет разбивать результат по дням — он возвращает только одно среднее число за весь период. Поэтому детализацию по дням я показать не могу.

Могу предложить два варианта:
1. Посчитать общее среднесуточное значение за первую неделю марта 2021 (1–7 марта) по дороге ЛСН.
2. Вызвать расчёт отдельно на каждый день (1, 2, 3, 4, 5, 6, 7 марта) — тогда получится 7 отдельных среднесуточных значений, по одному на день.

Какой вариант подойдёт?
